# Continuous-control (TD3) plots

Figures for the MuJoCo TD3 experiments. Run `run_td3.sh` (or `submit_td3_seeds`)
first to populate the output directory, then replace each
`'TODO: env_name run'` placeholder below with the directory holding that
environment's runs. Figures are saved to `plots/`.


In [ ]:
from collections import defaultdict
from matplotlib.transforms import Bbox
from tbparse import SummaryReader


In [ ]:
from typing import Dict, Any, List

import numpy as np
%load_ext autoreload
%autoreload 2

In [ ]:
#from environment_utils import *
from matplotlib import pyplot as plt
import matplotlib
from matplotlib.patches import Rectangle
import scipy.stats
from scipy.ndimage import uniform_filter1d
from scipy.stats import bootstrap
import re


import os
import dataclasses
import json
# from mpl_sizes import get_format

# formatter = get_format("NeurIPS") # options: ICLR, ICML, NeurIPS, InfThesis

blue = '#4882a6'
green = '#5eae94'
orange = '#e15e45'
purple = '#6D247A'
pink = '#AB1368'
yellow = '#F1C500'
grey = '#efefef'
light_blue = '#7DAED9'
brown = '#7f4f24'  # Freeze RM baseline

legend_color = '#ffffff'

In [ ]:
import os

@dataclasses.dataclass
class RunMetrics:
    steps: list
    performance: list
    returns: list
    sources: list

    def append(self, df, source='unknown'):
        self.steps.append(df['step'])
        self.performance.append(df['charts/eval_performance'])
        self.returns.append(df['charts/eval_return'])
        self.sources.append(source)

    def stack(self, run_key='unknown'):
        if not self.steps:
            return
        step_shapes = [np.asarray(arr).shape for arr in self.steps]
        performance_shapes = [np.asarray(arr).shape for arr in self.performance]
        return_shapes = [np.asarray(arr).shape for arr in self.returns]
        if len(set(step_shapes)) > 1 or len(set(performance_shapes)) > 1 or len(set(return_shapes)) > 1:
            print(f"[shape-mismatch] category={run_key}")
            for idx, (source, step_shape, perf_shape, ret_shape) in enumerate(zip(self.sources, step_shapes, performance_shapes, return_shapes)):
                print(
                    f"  idx={idx} source={source} step_shape={step_shape} "
                    f"perf_shape={perf_shape} return_shape={ret_shape}"
                )
            raise ValueError(
                f"Cannot stack category '{run_key}': mismatched shapes "
                f"steps={step_shapes} performance={performance_shapes} returns={return_shapes}"
            )
        self.steps = np.stack(self.steps)
        self.performance = np.stack(self.performance)
        self.returns = np.stack(self.returns)


In [ ]:
from dataclasses import field
import numpy as np
import dataclasses
from matplotlib.lines import Line2D

@dataclasses.dataclass
class PlotConfig():
    x_min: float = 0
    x_max: float = 1
    y_min: float = 0
    y_max: float = 1
    show_legend: bool = False
    show_x_label: bool = False
    show_y_label: bool = False
    y_ticks: np.array = field(default_factory=lambda: np.zeros(1))
    true_performance_y_min: float = None
    true_performance_y_max: float = None
    true_performance_y_ticks: np.array = None
    observed_return_y_min: float = None
    observed_return_y_max: float = None
    observed_return_y_ticks: np.array = None
    x_ticks: np.array = None
    show_x_ticks: bool = True
    env_name: str = 'NotDefined'
    show_divider: bool = True
    divider_label: str = 'Switch to $\mathit{Full}$'
    smoothing: int = 1
    legend_ncols: int = 4
    save_legend_separately: bool = False
    legend_offset_y: float = 0
    legend_offset_x: float = 0
    legend_columnspacing: float = 0
    override_with_env_defaults: bool = True
    vertical_layout: bool = False


def mean_confidence_interval(data, confidence=0.95):
    # compute mean and confidence interval using scipy.stats.bootstrap
    m = np.mean(data, axis=0)
    data = (data,)
    bootstrap_ci = bootstrap(data, 
                             statistic=np.mean,
                             n_resamples=1000, 
                             confidence_level=confidence,
                             method='percentile',
                             axis=0).confidence_interval
    return m, bootstrap_ci.low, bootstrap_ci.high

def smoothen(x, config):
    return uniform_filter1d(x, size=config.smoothing)

def resolve_y_axis_config(config, axis_name):
    if axis_name == 'true_performance':
        y_min = config.true_performance_y_min
        y_max = config.true_performance_y_max
        y_ticks = config.true_performance_y_ticks
    elif axis_name == 'observed_return':
        y_min = config.observed_return_y_min
        y_max = config.observed_return_y_max
        y_ticks = config.observed_return_y_ticks
    else:
        raise ValueError(f'Unknown axis name: {axis_name}')
    if y_min is None:
        y_min = config.y_min
    if y_max is None:
        y_max = config.y_max
    if y_ticks is None:
        y_ticks = config.y_ticks
    return y_min, y_max, y_ticks

def format_step_tick(step):
    if isinstance(step, (float, np.floating)) and not float(step).is_integer():
        return f'{step:g}'
    step_int = int(step)
    if step_int == 0:
        return '0'
    if step_int % 1_000_000 == 0:
        return f'{step_int // 1_000_000}e6'
    if step_int % 100_000 == 0:
        return f'{step_int // 100_000}e5'
    return str(step_int)

def format_y_tick(value):
    if isinstance(value, (float, np.floating)) and not float(value).is_integer():
        return f'{value:g}'
    value_int = int(value)
    if value_int == 0:
        return '0'
    if abs(value_int) >= 1_000 and value_int % 1_000 == 0:
        return f'{value_int // 1_000}e3'
    return str(value_int)

def plot_line(ax, x, y, c, label, config):
    x = x[0]
    y_mean, y_cfm, y_cfp = mean_confidence_interval(y)
    # y_mean, y_cfm, y_cfp = y.mean(0), y.mean(0) - y.std(0),  y.mean(0) + y.std(0)
    y_mean, y_cfm, y_cfp = smoothen(y_mean, config), smoothen(y_cfm, config), smoothen(y_cfp, config)
    ax.plot(x, y_mean, c=c, label=label)
    ax.fill_between(x, y_cfm, y_cfp, color=c, alpha=.3)

def plot_metrics(axs, metrics, conf, init=None, col='r', label='', name=''):
    returns = metrics['eval_returns'].copy()
    performance = metrics['eval_performances'].copy()
    x = metrics['eval_steps'].copy()
    if init is not None and init.keys():
        x += init['eval_steps'][-1,-1]
        x = np.concatenate([init['eval_steps'][:, -1, None], x], axis=-1)
        returns = np.concatenate([init['eval_returns'][:, -1, None], returns], axis=-1)
        performance = np.concatenate([init['eval_performances'][:, -1, None], performance], axis=-1)
    elif conf.show_divider:
        for ax in axs:
            ax.axvline(x[-1,-1], linestyle='dashed', c='k', label=conf.divider_label, linewidth=1)
    plot_line(axs[1], x, returns, c=col, label=label, config=conf)
    plot_line(axs[0], x, performance, c=col, label=label, config=conf)
        
    for ax in axs:
        ax.set_xlim(conf.x_min, conf.x_max)
        ax.spines[['right', 'top', 'bottom']].set_visible(False)
        if conf.show_x_label:
            ax.set_xlabel('steps')
        ax.set_facecolor(grey)
        ax.grid(axis='y', color='white')
        if conf.x_ticks is not None:
            ax.set_xticks(conf.x_ticks)
            ax.set_xticklabels([format_step_tick(tick) for tick in conf.x_ticks])
    performance_y_min, performance_y_max, performance_y_ticks = resolve_y_axis_config(conf, 'true_performance')
    observed_return_y_min, observed_return_y_max, observed_return_y_ticks = resolve_y_axis_config(conf, 'observed_return')
    axs[0].set_ylim(performance_y_min, performance_y_max)
    if performance_y_ticks is not None:
        axs[0].set_yticks(performance_y_ticks)
        axs[0].set_yticklabels([format_y_tick(tick) for tick in performance_y_ticks])
    axs[1].set_ylim(observed_return_y_min, observed_return_y_max)
    if observed_return_y_ticks is not None:
        axs[1].set_yticks(observed_return_y_ticks)
        axs[1].set_yticklabels([format_y_tick(tick) for tick in observed_return_y_ticks])
    if not conf.show_x_ticks:
            axs[0].set_xticks([])
    if conf.vertical_layout:
        axs[0].set_xlabel(None)
    if conf.show_legend:
        handles, labels = ax.get_legend_handles_labels()
        handles = handles
        def atoi(text):
            return int(text) if text.isdigit() else text
        # sort both labels and handles by labels
        labels, handles = zip(*sorted(zip(labels, handles), key=lambda t: [ atoi(c) for c in re.split(r'(\d+)', t[0]) ]))
        legend_offset = -3.72 + conf.legend_offset_y
        if conf.show_x_label:
            legend_offset += -0.22
        yellow_line = Line2D([0], [0], label='TD-3', color=pink)
        purple_line = Line2D([0], [0], label='MC-TD3 (ours)', color=purple)
        handles = list(handles)
        labels = list(labels)
        handles.insert(2, yellow_line)
        handles.insert(3, purple_line)
        labels.insert(2, 'TD-3')
        labels.insert(3, 'MC-TD3 (ours)')
        legend = axs[0].legend(loc='lower center', bbox_to_anchor=(1.05 + conf.legend_offset_x, legend_offset),
          fancybox=True, shadow=False, ncol=conf.legend_ncols, handles=handles, labelspacing=0, columnspacing=1.5 + conf.legend_columnspacing, facecolor=legend_color, borderpad=0.3, edgecolor=legend_color)

        def export_legend(legend, filename=f"plots/{name}_legend.pdf"):
            fig  = legend.figure
            fig.canvas.draw()
            bbox  = legend.get_window_extent()
            bbox = bbox.from_extents(*(bbox.extents))
            bbox = bbox.transformed(fig.dpi_scale_trans.inverted())
            fig.savefig(filename, bbox_inches=bbox)
        
        if conf.save_legend_separately:
            export_legend(legend)
            legend.remove()
    
    # axs[0].set_title(f'{env_name} Episode Return')
    # axs[1].set_title(f'{env_name} Episode Performance')
    if conf.show_y_label:
        axs[1].set_ylabel('episode return', rotation=0, loc='top', labelpad=-58, fontsize=10)
        axs[0].set_ylabel('episode performance', rotation=0, loc='top', labelpad=-84, fontsize=10)
        
def plot_run(run, conf):
    plot_multirun([(run, 'MC-DDQN (ours)', green)], conf, name=run, tampering_label='DDQN')
    
    
def plot_multirun(runs_labels_colors, conf, name, tampering_label=None, tampering_color=orange):
    plt.rcParams["font.family"] = "Times New Roman"
    plt.rcParams["font.size"] = "10"
    plt.rcParams["font.serif"] = ["Times New Roman"]
    plt.rcParams['mathtext.fontset'] = 'custom'
    plt.rcParams['mathtext.rm'] = 'Times New Roman'
    plt.rcParams['mathtext.it'] = 'Times New Roman:italic'
    plt.tight_layout()


    fig_height = 1.1
    if conf.show_y_label:
        fig_height += 0.1
    if conf.show_x_label:
        fig_height += 0.1
    if conf.vertical_layout:
        fig_height = 3.5
    if conf.vertical_layout:
        fig, axs = plt.subplots(2, 1, figsize=(1.5, fig_height))
    else:
        fig, axs = plt.subplots(1, 2, figsize=(3.5, fig_height))
    fig.subplots_adjust(wspace=0.4)
    # fig, axs = plt.subplots(1, 2, figsize=(7, 3))
    for i, (run, label, color) in enumerate(runs_labels_colors):
        if conf.override_with_env_defaults:
            change_config_for_env(run, conf)
        with open(f'results/{run}/config.json', 'r') as f:
            d = json.load(f)
            
        initial_metrics, tampering_metrics, no_tampering_metrics = load_metrics(run, d)
        if 'eval_steps' not in initial_metrics.keys():
            init_steps = 0
        else:
            init_steps = initial_metrics['eval_steps'].max()
        conf.x_max = init_steps + no_tampering_metrics['eval_steps'].max()+2
        
        if i == len(runs_labels_colors) - 1:
            plot_metrics(axs, initial_metrics, conf, col=blue, label='Training in $\mathit{Safe}$', name=name)
            if tampering_label is not None:
                    plot_metrics(axs, tampering_metrics, conf, init=initial_metrics, col=tampering_color, label=tampering_label, name=name)
        plot_metrics(axs, no_tampering_metrics, conf, init=initial_metrics, col=color, label=label, name=name)
    fig.patch.set_facecolor('None')
    fig.savefig(f'plots/{name}.pdf', facecolor=fig.get_facecolor(), bbox_inches=Bbox.from_bounds(-0.16,-0.05,1.75,3.4))
    #fig.savefig(f'plots/{name}.pdf',bbox_inches='tight')
    plt.show()


plt.tight_layout()

In [ ]:
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["font.size"] = "10"
plt.rcParams["font.serif"] = ["Times New Roman"]
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.rm'] = 'Times New Roman'
plt.rcParams['mathtext.it'] = 'Times New Roman:italic'

import hashlib
import pickle
from concurrent.futures import ThreadPoolExecutor, as_completed

SCALARS_CACHE_DIR = os.path.join('.cache', 'plots_td3_scalars')
os.makedirs(SCALARS_CACHE_DIR, exist_ok=True)
RUN_SCALARS_MEMORY_CACHE = {}

def _event_file_stats(run_dir):
    if os.path.isfile(run_dir):
        candidates = [run_dir]
    else:
        candidates = []
        for root, _, files in os.walk(run_dir):
            for fname in files:
                if 'tfevents' in fname:
                    candidates.append(os.path.join(root, fname))
    if not candidates and os.path.exists(run_dir):
        candidates = [run_dir]
    stats = []
    for path in sorted(candidates):
        try:
            st = os.stat(path)
        except OSError:
            continue
        rel_path = os.path.relpath(path, run_dir) if os.path.isdir(run_dir) else os.path.basename(path)
        stats.append((rel_path, st.st_size, st.st_mtime_ns))
    return stats

def _run_signature(run_dir):
    stats = _event_file_stats(run_dir)
    hasher = hashlib.sha1()
    hasher.update(os.path.abspath(run_dir).encode('utf-8'))
    for rel_path, size, mtime_ns in stats:
        hasher.update(rel_path.encode('utf-8'))
        hasher.update(str(size).encode('utf-8'))
        hasher.update(str(mtime_ns).encode('utf-8'))
    return hasher.hexdigest()

def _cache_path_for_run(run_dir):
    run_key = hashlib.sha1(os.path.abspath(run_dir).encode('utf-8')).hexdigest()
    return os.path.join(SCALARS_CACHE_DIR, f'{run_key}.pkl')

def clear_scalars_cache(memory_only=False):
    RUN_SCALARS_MEMORY_CACHE.clear()
    if memory_only:
        return
    if not os.path.isdir(SCALARS_CACHE_DIR):
        return
    for fname in os.listdir(SCALARS_CACHE_DIR):
        if fname.endswith('.pkl'):
            os.remove(os.path.join(SCALARS_CACHE_DIR, fname))

def read_eval_scalars(run_dir, use_cache=True, refresh_cache=False, verbose=False, use_pivot_reader=False):
    required_cols = ['step', 'charts/eval_performance', 'charts/eval_return']
    required_tags = ['charts/eval_performance', 'charts/eval_return']
    signature = _run_signature(run_dir)
    abs_run_dir = os.path.abspath(run_dir)
    memory_key = (abs_run_dir, signature)
    if use_cache and not refresh_cache and memory_key in RUN_SCALARS_MEMORY_CACHE:
        if verbose:
            print('memory cache', run_dir)
        return RUN_SCALARS_MEMORY_CACHE[memory_key].copy()

    cache_path = _cache_path_for_run(run_dir)
    if use_cache and not refresh_cache and os.path.exists(cache_path):
        try:
            with open(cache_path, 'rb') as f:
                payload = pickle.load(f)
            if payload.get('signature') == signature:
                df = payload['df']
                RUN_SCALARS_MEMORY_CACHE[memory_key] = df
                if verbose:
                    print('disk cache', run_dir)
                return df.copy()
        except Exception:
            pass

    if use_pivot_reader:
        reader = SummaryReader(run_dir, pivot=True)
        df = reader.scalars
    else:
        reader = SummaryReader(run_dir, pivot=False)
        long_df = reader.scalars
        required_long_cols = {'step', 'tag', 'value'}
        if not required_long_cols.issubset(long_df.columns):
            raise KeyError(f'Missing columns {sorted(required_long_cols - set(long_df.columns))} in {run_dir}')
        long_df = long_df[long_df['tag'].isin(required_tags)][['step', 'tag', 'value']]
        if long_df.empty:
            raise KeyError(f'Missing required tags {required_tags} in {run_dir}')
        df = long_df.pivot_table(index='step', columns='tag', values='value', aggfunc='last').reset_index()
        df.columns.name = None
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise KeyError(f'Missing columns {missing} in {run_dir}')
    df = df[required_cols].dropna().reset_index(drop=True)

    if use_cache:
        RUN_SCALARS_MEMORY_CACHE[memory_key] = df
        try:
            with open(cache_path, 'wb') as f:
                pickle.dump({'signature': signature, 'df': df}, f, protocol=pickle.HIGHEST_PROTOCOL)
        except Exception:
            pass
    if verbose:
        print('parsed', run_dir)
    return df.copy()

def _classify_run_key(file_name):
    if '__True' in file_name or 'CheckTamperingTrue' in file_name:
        return 'no_tampering'
    if 'oracle' in file_name:
        return 'oracle'
    if 'frozen' in file_name:
        return 'frozen'
    if 'relabelRM' in file_name:
        # Frozen reward-model (No-Gate) runs: bucket separately so they do
        # not fall into the 'tampering' curve.
        return 'no_gate'
    return 'tampering'

def plot_environment(
    run_folder,
    save_name=None,
    conf=None,
    use_cache=True,
    refresh_cache=False,
    verbose_reads=True,
    parallel_reads=True,
    max_workers=None,
    use_pivot_reader=False,
):
    runs = {
        "tampering": RunMetrics([], [], [], []),
        "no_tampering": RunMetrics([], [], [], []),
        "oracle": RunMetrics([], [], [], []),
        "frozen": RunMetrics([], [], [], []),
        "no_gate": RunMetrics([], [], [], []),
    }
    run_entries = []
    for file in sorted(os.listdir(run_folder)):
        run_dir = os.path.join(run_folder, file)
        if os.path.isdir(run_dir) or os.path.isfile(run_dir):
            run_entries.append((file, run_dir))

    if parallel_reads and len(run_entries) > 1:
        workers = max_workers
        if workers is None:
            workers = min(40, len(run_entries))
        with ThreadPoolExecutor(max_workers=workers) as executor:
            futures = {
                executor.submit(
                    read_eval_scalars,
                    run_dir,
                    use_cache,
                    refresh_cache,
                    verbose_reads,
                    use_pivot_reader,
                ): (file, run_dir)
                for file, run_dir in run_entries
            }
            for future in as_completed(futures):
                file, run_dir = futures[future]
                try:
                    df = future.result()
                except Exception:
                    if verbose_reads:
                        print('failed to read', run_dir)
                    continue
                key = _classify_run_key(file)
                runs[key].append(df, source=file)
    else:
        for file, run_dir in run_entries:
            try:
                df = read_eval_scalars(run_dir, use_cache=use_cache, refresh_cache=refresh_cache, verbose=verbose_reads, use_pivot_reader=use_pivot_reader)
            except Exception:
                if verbose_reads:
                    print('failed to read', run_dir)
                continue
            key = _classify_run_key(file)
            runs[key].append(df, source=file)

    for key in runs:
        runs[key].stack(run_key=key)

    fig_height = 3.5
    fig, axs = plt.subplots(2, 1, figsize=(1.5 / 1, fig_height))
    if conf is None:
        conf = PlotConfig(
            y_min=-30,
            y_max=35,
            x_ticks=[0, 100_000, 200_000],
            y_ticks=np.arange(-30, 31, 10),
            show_y_label=False,
            show_x_label=True,
            vertical_layout=True,
        )
        
    for ax in axs:
        ax.set_xlim(conf.x_min, conf.x_max)
        ax.spines[['right', 'top', 'bottom']].set_visible(False)
        if conf.show_x_label:
            ax.set_xlabel('steps')
        ax.set_facecolor(grey)
        ax.grid(axis='y', color='white')
        ax.set_xmargin(10)   
        if getattr(conf, 'x_ticks', None) is not None:
            ax.set_xticks(conf.x_ticks)
            ax.set_xticklabels([format_step_tick(tick) for tick in conf.x_ticks])
    performance_y_min, performance_y_max, performance_y_ticks = resolve_y_axis_config(conf, 'true_performance')
    observed_return_y_min, observed_return_y_max, observed_return_y_ticks = resolve_y_axis_config(conf, 'observed_return')
    axs[0].set_ylim(performance_y_min, performance_y_max)
    if performance_y_ticks is not None:
        axs[0].set_yticks(performance_y_ticks)
        axs[0].set_yticklabels([format_y_tick(tick) for tick in performance_y_ticks])
    axs[1].set_ylim(observed_return_y_min, observed_return_y_max)
    if observed_return_y_ticks is not None:
        axs[1].set_yticks(observed_return_y_ticks)
        axs[1].set_yticklabels([format_y_tick(tick) for tick in observed_return_y_ticks])
            
    if conf.vertical_layout:
        axs[0].set_xlabel(None)
    axs[0].set_xticks([])
    axs[1].set_xlabel('steps')

    def local_plot_run(run_metrics, color, label):
        if hasattr(run_metrics, 'steps') and len(run_metrics.steps) > 0:
            plot_line(axs[1], run_metrics.steps, run_metrics.returns, color, label, conf)
            plot_line(axs[0], run_metrics.steps, run_metrics.performance, color, label, conf)

    print('plotting frozen')
    local_plot_run(runs['frozen'], light_blue, 'Frozen')
    print('plotting oracle')
    local_plot_run(runs['oracle'], yellow, 'Oracle')
    print('plotting tampering')
    local_plot_run(runs['tampering'], pink, 'TD3')
    print('plotting no tampering')
    local_plot_run(runs['no_tampering'], purple, 'MC-TD3 (ours)')

    if save_name is None:
        save_name = os.path.basename(os.path.normpath(run_folder))
    
    os.makedirs('plots', exist_ok=True)
    fig.patch.set_facecolor('None')
    fig.patch.set_alpha(0)
    fig.savefig(f'plots/{save_name}.pdf', facecolor=fig.get_facecolor(), bbox_inches='tight')
    plt.show()
    
# Example call:
# conf = PlotConfig(...)
# plot_environment("runs", save_name="td3", conf=conf)


In [ ]:
conf = PlotConfig(
    y_min=-35,
    y_max=35,
    x_ticks=[0, 100_000, 200_000],
    y_ticks=np.arange(-30, 31, 10),
    show_y_label=False,
    show_x_label=True,
    vertical_layout=True,
    true_performance_y_min=-35,
    true_performance_y_max=0,
    true_performance_y_ticks=np.arange(-30, 0, 5),
)

plot_environment('TODO: env_name run', save_name='td3_reacher', conf=conf)

In [ ]:
conf = PlotConfig(
    x_ticks=[0, 500_000, 1_000_000],
    show_y_label=False,
    show_x_label=True,
    vertical_layout=True,
    true_performance_y_min=-1300,
    true_performance_y_max=4200,
    true_performance_y_ticks=np.arange(-1000, 4001, 1000),
    observed_return_y_min=-1300,
    observed_return_y_max=10_000,
    observed_return_y_ticks = np.arange(-1000, 10001, 2000)
)

plot_environment('TODO: env_name run', save_name='td3_ant', conf=conf)

In [ ]:
conf = PlotConfig(
    x_ticks=[0, 500_000, 1_000_000],
    show_y_label=False,
    show_x_label=True,
    vertical_layout=True,
    true_performance_y_min=-1300,
    true_performance_y_max=11000,
    true_performance_y_ticks=np.arange(0, 11000, 2000),
    observed_return_y_min=-1300,
    observed_return_y_max=29_000,
    observed_return_y_ticks = np.arange(0, 29_000, 5000)
)

plot_environment('TODO: env_name run', save_name='td3_cheetah', conf=conf)

In [ ]:
blue = '#4882a6'
green = '#5eae94'
orange = '#e15e45'
purple = '#6D247A'
pink = '#AB1368'
yellow = '#F1C500'
grey = '#efefef'
light_blue = '#7DAED9'

colors = [orange, green, pink, purple, yellow, light_blue, blue]
lines = [Line2D([0], [0], color=c, linewidth=1) for c in colors]
labels = ['DDQN', 'MC-DDQN (ours)', 'TD3', 'MC-TD3 (ours)', 'Oracle', 'Frozen', 'Pretraining']
fig, ax = plt.subplots()

# Hide the axes to prevent any plot elements from being displayed
ax.axis('off')

# Create the legend using the custom handles and labels
ax.legend(lines, labels, fancybox=True, shadow=False, ncol=7, labelspacing=0, columnspacing=1, facecolor=legend_color, borderpad=0.3, edgecolor=legend_color, handlelength=1)

plt.savefig('plots/td3_legend.pdf', bbox_inches='tight')
plt.show()